# 04 Google Colab で CPU・GPU・TPU を比較する（ANN と Spiking Manifesto 型 LUT）

## 1. 研究目的

ANN の forward と Spiking Manifesto 型 LUT の forward を、**同じ入力 shape・同じ計算内容**で CPU・GPU・TPU 上で実行し、実行時間を比べます。

理解したいこと：

- CPU と GPU の違い、GPU と TPU の違い
- **行列演算**（ANN）と、**比較演算・LUT 参照・不規則なメモリアクセス**（LUT）の違い
- **バッチ処理**（多くの入力をまとめて計算する）の効果
- 実行時間測定の注意点（**非同期実行と同期**、**初回コンパイル**、**データ転送**）

> ### ⚠️ 最初に読む
> - GPU や TPU は「速い CPU」ではありません。**小さな計算では CPU の方が速いこともあります**。
> - LUT は行列積ではないので、行列演算に特化したハードウェア上で LUT が速くなるとは限りません。
> - **Colab の CPU・GPU・TPU の処理時間比較から、SNN 専用ニューロモルフィックハードウェアの電力効率を結論することはできません。** 消費電力は測っていません。
> - このセルの実行結果に出てくる数値は、**あなたが実際に使ったランタイムでの測定値だけ**です。使えなかったデバイスの値は表示しません。

## 2. 理論的背景

| デバイス | 特徴 | 得意な計算 |
| --- | --- | --- |
| **CPU** | 少数の高性能コア。分岐・逐次処理・不規則なアクセスに強い。起動の遅延が小さい | 小さな計算、分岐の多い処理 |
| **GPU**（NVIDIA、CUDA） | 数千の単純なコアで同じ計算を一斉に行う。命令は**非同期**に実行される | 大きな行列積、要素ごとの演算をまとめて大量に |
| **TPU**（Google） | 行列積専用の回路（systolic array）を持つ。**XLA コンパイラ**で計算全体をコンパイルしてから実行する。**shape が固定**されている計算が前提 | 大きな行列積（特に bfloat16） |

### ANN と LUT の計算の形

| | ANN | LUT |
| --- | --- | --- |
| 中心の演算 | 行列積 $XW^\top$（規則的、再利用が多い） | 比較 $X_{:,a} > X_{:,b}$、整数化、**gather（index による行の読み出し）**、和 |
| メモリアクセス | 連続的・規則的 | index に依存する**不規則な**アクセス |
| GPU・TPU との相性 | 非常に良い | 比較・和は並列化できるが、gather は行列積ほど効率よく使えないことがある |

### Tensor による LUT の実装（`tensor_backends.py`）

Python の for ループを GPU で回すのではなく、バッチ全体を Tensor 演算で計算します。

```text
X        : (B, n)          入力の latency vector を B 個まとめたもの
anchors  : (nt, nc, 2)     各 LUT の anchor pair (a, b)
rows     : (nt, 2^nc, m)   各 LUT の行
bits     = X[:, a] > X[:, b]              → (B, nt, nc)
index    = Σ_r bits[..., r] · 2^(nc-1-r)  → (B, nt)      1 つ目の比較が最上位ビット
selected = rows[t, index[:, t]]           → (B, nt, m)   gather
Y        = Σ_t selected[:, t, :]          → (B, m)
```
これは `spike_lut.lut_forward`（1 入力ずつ for ループ）と**同じ結果**になることを、下で確かめます。

## 参考にした Notebook

授業の `ai-programming-2026-team-project-1-starter` の `A_cpu_gpu_tpu/A_cpu_gpu_tpu_benchmark/starter.ipynb`（MIT License）を参考に、次の点を取り入れ、E3 向けに改変しました。

- `torch.cuda.is_available()` による GPU の検出、`torch.cuda.synchronize()` で同期してから時間を測る方法
- 利用できないデバイスは `try` / `if` で**スキップ**して続ける構成、結果を棒グラフにする流れ
- 変更点：TPU は参考 Notebook の TensorFlow（`TPUStrategy`）ではなく **JAX**（`jax.jit`、`block_until_ready()`）を使う。1 回だけの測定ではなく**ウォームアップ＋複数回の中央値**にし、初回・コンパイル・転送を**分けて**測る

## 3. 実験環境（最初に必ず実行）

このセルは、E3 リポジトリの `.py` ファイル（`ann.py`、`spike_lut.py` など）を import できるように、
**作業ディレクトリ（E3 のルート）を自動で探して設定**します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのチームのリポジトリの URL** を入れる（`REPO_SUBDIR` に E3 のフォルダ名） |
| Colab（方法B：ZIP をアップロード） | E3 フォルダを ZIP にして、Colab の左の「ファイル」に `/content` へアップロードしておく（またはこのセルの実行中にアップロード） |

> ⚠️ `textbook/` フォルダ（教科書のコード）は**公開リポジトリには含めない**約束です。
> 方法A で `textbook/` がない場合、`spike_encoding.py`・`benchmark.py` の import が失敗します。
> その場合は方法B（ZIP）を使うか、授業で配布された `textbook/` を同じ場所に置いてください。

In [ ]:
import os
import sys
import glob
import zipfile
import subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                           # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "e3-ann-snn-comparison"   # リポジトリの中の E3 フォルダ（ルートが E3 なら "" にする）

IN_COLAB = "google.colab" in sys.modules


def is_e3_root(p):
    """spike_lut.py と ann.py があるフォルダを E3 のルートとみなす"""
    return (Path(p) / "spike_lut.py").exists() and (Path(p) / "ann.py").exists()


def find_e3_root():
    # 1) ローカル：notebooks/ の中、または E3 のルートで開いた場合
    for p in [Path.cwd(), Path.cwd().parent]:
        if is_e3_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    # 2) Colab 方法A：clone
    if REPO_URL:
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_e3_root(cand):
            return cand
    # 3) Colab 方法B：/content にある ZIP を展開（なければアップロードを求める）
    zips = glob.glob("/content/*.zip")
    if not zips:
        from google.colab import files
        print("E3 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/e3_upload")
    for hit in glob.glob("/content/e3_upload/**/spike_lut.py", recursive=True):
        if is_e3_root(Path(hit).parent):
            return Path(hit).parent
    return None


E3_ROOT = find_e3_root()
if E3_ROOT is None:
    raise RuntimeError("E3 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(E3_ROOT)
if str(E3_ROOT) not in sys.path:
    sys.path.insert(0, str(E3_ROOT))
print("E3_ROOT:", E3_ROOT)
if not (E3_ROOT / "textbook" / "insertion_sort.py").exists():
    print("⚠️ textbook/ がありません。spike_encoding.py などの import が失敗します（上の注意を参照）。")

In [ ]:
import matplotlib.pyplot as plt

# グラフの共通スタイル（Colab には日本語フォントがないことが多いので、図の文字は英語にしている）
plt.rcParams.update({
    "figure.figsize": (7, 3.5), "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
})
# 色は役割ごとに固定する（ANN / LIF・SNN / LUT、CPU / GPU / TPU）
COLORS = {"ANN": "#2a78d6", "LIF": "#1baf7a", "LUT": "#eb6834",
          "CPU": "#2a78d6", "GPU": "#eb6834", "TPU": "#1baf7a", "neutral": "#8a8985"}

### ライブラリの準備

Colab には NumPy・PyTorch・JAX が通常インストールされています（ランタイムの種類により、JAX の GPU/TPU 対応版が入っています）。
ローカルで実行する場合、PyTorch・JAX がなければその部分はスキップされます（必要なら下のコメントを外してインストール）。

In [ ]:
# ローカル（CPU）で PyTorch・JAX も試したい場合だけ、コメントを外して実行する
# !pip install -q torch jax

### 3-1. 実行環境の検出

Python・PyTorch・JAX のバージョン、CPU、GPU・TPU の有無を表示します。**ここでは測定はしません。**

In [ ]:
import os
import time
import json
import platform
import numpy as np

import tensor_backends as tb
from ann import dense_forward
from spike_lut import lut_forward, make_random_tables

env = tb.detect_environment()

cpu_name = env["cpu"]
try:   # Linux（Colab を含む）では /proc/cpuinfo から CPU 名を読む
    with open("/proc/cpuinfo") as f:
        cpu_name = next(line.split(":", 1)[1].strip() for line in f if line.startswith("model name"))
except (OSError, StopIteration):
    pass

print("Python     :", env["python"])
print("Platform   :", env["platform"])
print("CPU        :", cpu_name, f"({os.cpu_count()} logical cores)")
print("NumPy      :", env["numpy"])
print("PyTorch    :", env["torch"] or "not installed")
print("CUDA GPU   :", env["gpu_name"] if env["cuda_available"] else "not available")
print("JAX        :", env["jax"] or "not installed")
print("JAX devices:", env["jax_devices"] or "-")
print("TPU        :", "available" if env["tpu_available"] else "not available")

### 3-2. 実験条件と安全上限

Colab の無料ランタイムでも短時間で終わる、小さめの条件を初期値にしています。
`MAX_BYTES` を超える条件、`TIME_BUDGET_S` を超えたあとの条件は**実行せずにスキップ**します。

In [ ]:
SEED = 0
DTYPE = np.float32           # すべてのデバイスで float32 を使う（結果に記録する）
WARMUP, REPEATS = 3, 20      # ウォームアップ回数と測定回数（中央値を使う）

BASE = dict(n=64, m=64, nt=16, nc=6)        # 基本の問題規模
BATCH_SIZES = [1, 32, 256, 1024]            # バッチサイズの比較
INPUT_DIMS = [16, 64, 256]                  # 入力次元の比較（m = n、バッチ 256）
SWEEP_BATCH = 256

MAX_BYTES = 256 * 1024**2    # 1 条件で使う配列の大きさの上限（約 256 MB）
TIME_BUDGET_S = 300          # 全測定の時間の目安（秒）。超えたら残りをスキップ
PY_REF_MAX_BATCH = 32        # 純 Python の基準実装はとても遅いので、このバッチまでだけ測る


def estimated_bytes(B, n, m, nt, nc, itemsize=4):
    """1 条件で作る主な配列のバイト数の見積もり（入力・重み・LUT 行・中間結果）"""
    return itemsize * (B * n + n * m + m + nt * 2**nc * m + B * nt * nc * 3 + B * nt * m + B * m)

## 4. 使用するコード

| ファイル | 役割 |
| --- | --- |
| `ann.py`・`spike_lut.py` | **基準実装**（純 Python、1 入力ずつ）。正しさの基準 |
| `benchmark.py` | `make_ann`：seed 固定の ANN の重み |
| `tensor_backends.py` | NumPy・PyTorch・JAX の**バッチ版**、環境検出、`time_call`（ウォームアップ・同期つき測定） |

### 測る時間の区別

| 記録名 | 意味 |
| --- | --- |
| `param_transfer_s` | 重み・LUT を CPU のメモリからデバイスへ送る時間（1 回だけ） |
| `input_transfer_s` | 入力バッチ $X$ をデバイスへ送る時間 |
| `compile_s` | JAX の `jit` のコンパイル時間（`lower().compile()`） |
| `first_s` | 最初の 1 回の実行時間（初期化・コンパイル・キャッシュの影響を含み得る） |
| `warm_median_s` | ウォームアップ後の**計算本体**の実行時間（中央値） |

GPU・TPU は命令を**非同期**に実行します。PyTorch では `torch.cuda.synchronize()`、JAX では `block_until_ready()` で**計算の終了を待ってから**時計を止めます。

In [ ]:
results = []          # 測定結果（1 条件 1 行の辞書）。実際に測ったものだけを入れる
T_START = time.perf_counter()


def budget_left():
    return time.perf_counter() - T_START < TIME_BUDGET_S


def make_case(B, n, m, nt, nc, seed=SEED):
    """同じ seed から、全デバイス共通の入力・ANN の重み・LUT を作る（NumPy 配列）"""
    X = tb.make_batch(B, n, seed=seed, dtype=DTYPE)
    W, b = tb.make_ann_arrays(n, m, seed=seed, dtype=DTYPE)
    tables = make_random_tables(n, nt, nc, m, seed=seed)
    anchors, rows = tb.tables_to_arrays(tables, dtype=DTYPE)
    return dict(X=X, W=W, b=b, tables=tables, anchors=anchors, rows=rows)


def record(device, framework, model, B, n, m, nt, nc, timing, **extra):
    row = dict(device=device, framework=framework, model=model, batch=B, n=n, m=m, nt=nt, nc=nc,
               dtype=np.dtype(DTYPE).name, first_s=timing["first"], warm_median_s=timing["median"])
    row.update(extra)
    results.append(row)
    return row

## 5. 実行例

### 5-1. CPU：基準実装（純 Python）と NumPy 版

まず、既存の `ann.dense_forward`・`spike_lut.lut_forward` を 1 入力ずつ実行し、NumPy のバッチ版と**結果が一致する**ことを確かめます。

In [ ]:
case = make_case(32, **BASE)
X, W, b, tables, anchors, rows = (case[k] for k in ["X", "W", "b", "tables", "anchors", "rows"])

# 基準実装（純 Python のリスト）で 1 入力ずつ計算
X_list, W_list, b_list = X.tolist(), W.tolist(), b.tolist()
ref_ann = np.array([dense_forward(x, W_list, b_list) for x in X_list])
ref_lut = [lut_forward(x, tables) for x in X_list]

# NumPy のバッチ版
np_ann = tb.numpy_ann_forward(X, W, b)
np_idx = tb.numpy_lut_indices(X, anchors)
np_lut = tb.numpy_lut_forward(X, anchors, rows)

assert np_idx.tolist() == [r["indices"] for r in ref_lut], "LUT index が基準実装と一致しません"
np.testing.assert_allclose(np_lut, [r["output"] for r in ref_lut], rtol=1e-5, atol=1e-5)
np.testing.assert_allclose(np_ann, ref_ann, rtol=1e-5, atol=1e-5)
print("NumPy batch == pure-Python reference (LUT index: exact, outputs: rtol=1e-5)")

### 5-2. PyTorch（CPU と、使えれば CUDA GPU）

同じ配列を PyTorch の Tensor に変換し、`tensor_backends.torch_ann_forward`・`torch_lut_forward` で計算します。
GPU がないランタイムでは、GPU の部分は**スキップ**されます。

In [ ]:
try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
    print("PyTorch がないので、PyTorch の測定はスキップします")

TORCH_DEVICES = []
if HAS_TORCH:
    TORCH_DEVICES.append(("CPU", torch.device("cpu")))
    if torch.cuda.is_available():
        TORCH_DEVICES.append(("GPU", torch.device("cuda")))
        # TF32（精度を下げて速くする機能）を使わないことを明示して、float32 で比べる
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
    print("PyTorch devices:", [name for name, _ in TORCH_DEVICES])


def torch_sync(dev):
    """GPU なら計算の終了を待つ関数、CPU なら None"""
    return (lambda _r: torch.cuda.synchronize()) if dev.type == "cuda" else None


def to_device_timed(arrays, dev, dtype_map):
    """NumPy 配列を Tensor にしてデバイスへ送り、かかった時間（同期つき）を返す"""
    sync = torch_sync(dev)
    t0 = time.perf_counter()
    out = [torch.as_tensor(a).to(dev, dtype=dt) for a, dt in zip(arrays, dtype_map)]
    if sync:
        sync(None)
    return out, time.perf_counter() - t0


def bench_torch(case, label, dev):
    """1 つの条件で、PyTorch の ANN・LUT を測定し、NumPy 版と一致するか確かめる"""
    X, W, b, anchors, rows = (case[k] for k in ["X", "W", "b", "anchors", "rows"])
    B, n = X.shape; m = W.shape[0]; nt, nc = anchors.shape[:2]
    weights = tb.bit_weights(nc)
    (Wd, bd, Ad, Rd, wd), t_param = to_device_timed(
        [W, b, anchors, rows, weights], dev, [torch.float32, torch.float32, torch.long, torch.float32, torch.long])
    (Xd,), t_input = to_device_timed([X], dev, [torch.float32])
    sync = torch_sync(dev)
    mem = {}
    if dev.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    with torch.no_grad():
        t_ann = tb.time_call(lambda: tb.torch_ann_forward(Xd, Wd, bd), sync, WARMUP, REPEATS)
        t_lut = tb.time_call(lambda: tb.torch_lut_forward(Xd, Ad, Rd, wd), sync, WARMUP, REPEATS)
        # 正しさの確認（NumPy 版と比べる。index は完全一致、出力は許容誤差つき）
        idx = tb.torch_lut_indices(Xd, Ad, wd).cpu().numpy()
        assert (idx == tb.numpy_lut_indices(X, anchors)).all()
        np.testing.assert_allclose(tb.torch_lut_forward(Xd, Ad, Rd, wd).cpu().numpy(),
                                   tb.numpy_lut_forward(X, anchors, rows), rtol=1e-5, atol=1e-5)
        np.testing.assert_allclose(tb.torch_ann_forward(Xd, Wd, bd).cpu().numpy(),
                                   tb.numpy_ann_forward(X, W, b), rtol=1e-4, atol=1e-4)
    if dev.type == "cuda":
        mem["peak_device_bytes"] = torch.cuda.max_memory_allocated()
    common = dict(param_transfer_s=t_param, input_transfer_s=t_input, compile_s=None, **mem)
    record(label, "PyTorch", "ANN", B, n, m, nt, nc, t_ann, **common)
    record(label, "PyTorch", "LUT", B, n, m, nt, nc, t_lut, **common)


for label, dev in TORCH_DEVICES:
    bench_torch(case, label, dev)
    print(f"PyTorch {label}: results match NumPy (B=32)")

### 5-3. JAX（CPU・GPU・TPU のうち、ランタイムで使えるもの）

JAX は `jax.jit` で関数を**コンパイル**してから実行します（TPU では XLA コンパイルが必須）。

- **コンパイル時間**：`jax.jit(f).lower(...).compile()` にかかる時間。shape が変わるたびにコンパイルし直します（**静的 shape**）。
- **計算本体**：コンパイル済みの関数を実行し、`block_until_ready()` で終了を待つまでの時間。
- TPU では float32 の行列積が既定で低い精度（bfloat16 を使う計算）になることがあるため、`jax_default_matmul_precision` を `"highest"` にして float32 相当で比べます。

> ⚠️ **TPU での実行は、この教材の作成時には検証していません**（CPU 上の JAX でのみ動作確認）。TPU ランタイムで実行したら、結果とエラーの有無を記録してください。

In [ ]:
try:
    import jax
    import jax.numpy as jnp
    HAS_JAX = True
except ImportError:
    HAS_JAX = False
    print("JAX がないので、JAX の測定はスキップします")

if HAS_JAX:
    try:
        jax.config.update("jax_default_matmul_precision", "highest")
        JAX_MATMUL_PRECISION = "highest"
    except Exception as e:   # バージョンにより設定できない場合は記録して続ける
        JAX_MATMUL_PRECISION = f"default ({e})"
    JAX_DEVICE = jax.devices()[0]
    JAX_LABEL = {"tpu": "TPU", "gpu": "GPU", "cuda": "GPU", "cpu": "CPU"}.get(JAX_DEVICE.platform, JAX_DEVICE.platform)
    jax_ann, jax_lut = tb.jax_functions()
    print("JAX device:", JAX_DEVICE, "→ label", JAX_LABEL, "/ matmul precision:", JAX_MATMUL_PRECISION)


def jax_put_timed(arrays):
    t0 = time.perf_counter()
    out = [jax.device_put(a, JAX_DEVICE) for a in arrays]
    for o in out:
        o.block_until_ready()
    return out, time.perf_counter() - t0


def bench_jax(case):
    X, W, b, anchors, rows = (case[k] for k in ["X", "W", "b", "anchors", "rows"])
    B, n = X.shape; m = W.shape[0]; nt, nc = anchors.shape[:2]
    (Wd, bd, Ad, Rd), t_param = jax_put_timed([W, b, anchors, rows])
    (Xd,), t_input = jax_put_timed([X])
    sync = lambda r: r.block_until_ready()

    for model, f, args in [("ANN", jax_ann, (Xd, Wd, bd)), ("LUT", jax_lut, (Xd, Ad, Rd))]:
        # JAX は同じ関数のコンパイル結果をキャッシュするので、毎回「新しい関数オブジェクト」で包んで測る
        # (1) jit してすぐ呼ぶ「本当の初回」（トレース＋コンパイル＋実行）
        t0 = time.perf_counter()
        jax.jit(lambda *a: f(*a))(*args).block_until_ready()
        t_first_with_compile = time.perf_counter() - t0
        # (2) コンパイルだけ（実行しない）
        t0 = time.perf_counter()
        compiled = jax.jit(lambda *a: f(*a)).lower(*args).compile()
        t_compile = time.perf_counter() - t0
        # (3) コンパイル済みの関数の初回と、ウォームアップ後の中央値
        timing = tb.time_call(lambda: compiled(*args), sync, WARMUP, REPEATS)
        out = np.asarray(compiled(*args))
        if model == "LUT":
            np.testing.assert_allclose(out, tb.numpy_lut_forward(X, anchors, rows), rtol=1e-5, atol=1e-5)
        else:
            np.testing.assert_allclose(out, tb.numpy_ann_forward(X, W, b), rtol=1e-4, atol=1e-4)
        mem = {}
        try:
            stats = JAX_DEVICE.memory_stats()   # TPU・GPU では使えることが多い。CPU では None のことがある
            if stats and "peak_bytes_in_use" in stats:
                mem["peak_device_bytes"] = stats["peak_bytes_in_use"]
        except Exception:
            pass
        record(JAX_LABEL, "JAX", model, B, n, m, nt, nc, timing, param_transfer_s=t_param,
               input_transfer_s=t_input, compile_s=t_compile,
               first_with_compile_s=t_first_with_compile, **mem)
    # LUT index の完全一致も確認する
    idx_fn = jax.jit(lambda X_, A_: (((X_[:, A_[..., 0]] > X_[:, A_[..., 1]]).astype(jnp.int32))
                                     * (2 ** jnp.arange(A_.shape[1] - 1, -1, -1, dtype=jnp.int32))).sum(-1))
    assert (np.asarray(idx_fn(Xd, Ad)) == tb.numpy_lut_indices(X, anchors)).all()


if HAS_JAX:
    bench_jax(case)
    print(f"JAX {JAX_LABEL}: results match NumPy (B=32)")

### 5-4. CPU の NumPy 版と純 Python 版も、同じ形式で記録する

NumPy は CPU 上のバッチ計算の基準、純 Python は「1 入力ずつ for ループ」の基準です（純 Python は遅いのでバッチ `PY_REF_MAX_BATCH` まで）。

In [ ]:
def bench_cpu(case):
    X, W, b, tables, anchors, rows = (case[k] for k in ["X", "W", "b", "tables", "anchors", "rows"])
    B, n = X.shape; m = W.shape[0]; nt, nc = anchors.shape[:2]
    none = dict(param_transfer_s=None, input_transfer_s=None, compile_s=None)
    record("CPU", "NumPy", "ANN", B, n, m, nt, nc,
           tb.time_call(lambda: tb.numpy_ann_forward(X, W, b), None, WARMUP, REPEATS), **none)
    record("CPU", "NumPy", "LUT", B, n, m, nt, nc,
           tb.time_call(lambda: tb.numpy_lut_forward(X, anchors, rows), None, WARMUP, REPEATS), **none)
    if B <= PY_REF_MAX_BATCH:
        Xl, Wl, bl = X.tolist(), W.tolist(), b.tolist()
        record("CPU", "Python", "ANN", B, n, m, nt, nc,
               tb.time_call(lambda: [dense_forward(x, Wl, bl) for x in Xl], None, 1, 5), **none)
        record("CPU", "Python", "LUT", B, n, m, nt, nc,
               tb.time_call(lambda: [lut_forward(x, tables) for x in Xl], None, 1, 5), **none)


bench_cpu(case)
print(len(results), "rows recorded so far")

### 5-5. 問題規模を変えた測定（バッチサイズ・入力次元）

すべての組合せではなく、次の 2 系列だけを測ります（安全上限つき）。

1. バッチサイズ $B$ を変える（$n=m=64$、$n_t=16$、$n_c=6$）
2. 入力次元 $n$（$=m$）を変える（$B=256$）

In [ ]:
def run_all(B, n, m, nt, nc):
    if estimated_bytes(B, n, m, nt, nc) > MAX_BYTES:
        print(f"skip (memory limit): B={B}, n={n}, m={m}, nt={nt}, nc={nc}")
        return
    if not budget_left():
        print(f"skip (time budget): B={B}, n={n}")
        return
    c = make_case(B, n, m, nt, nc)
    bench_cpu(c)
    for label, dev in TORCH_DEVICES:
        bench_torch(c, label, dev)
    if HAS_JAX:
        bench_jax(c)

results.clear()          # 5-1〜5-4 の確認用の行を消して、ここから記録し直す
T_START = time.perf_counter()
for B in BATCH_SIZES:
    run_all(B, **BASE)
for d in INPUT_DIMS:
    if d != BASE["n"]:   # n = 64、B = 256 は上で測定済み
        run_all(SWEEP_BATCH, d, d, BASE["nt"], BASE["nc"])
print(f"{len(results)} rows, {time.perf_counter() - T_START:.1f} s")

### 5-6. 結果の表と保存

結果を CSV に保存します。Colab ではランタイムを切り替えると**ファイルが消える**ので、ダウンロードするか Google Drive に保存してください。
ランタイムごと（CPU・GPU・TPU）に別のファイル名で保存し、あとで結合して比較します。

In [ ]:
import csv

runtime_tag = "tpu" if env["tpu_available"] else ("gpu" if env["cuda_available"] else "cpu")
os.makedirs("notebooks/results", exist_ok=True)
out_path = f"notebooks/results/04_benchmark_{runtime_tag}.csv"
fields = sorted({k for r in results for k in r})
with open(out_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(results)
print("saved:", out_path)

# 簡単な表示（バッチ 256、n = 64）
print(f"{'device':6} {'framework':9} {'model':5} {'first [ms]':>11} {'warm median [ms]':>17} {'compile [ms]':>13}")
for r in results:
    if r["batch"] == 256 and r["n"] == BASE["n"]:
        comp = f"{r['compile_s'] * 1e3:13.2f}" if r.get("compile_s") is not None else f"{'-':>13}"
        print(f"{r['device']:6} {r['framework']:9} {r['model']:5} {r['first_s'] * 1e3:11.3f} "
              f"{r['warm_median_s'] * 1e3:17.4f} {comp}")

# Colab でダウンロードする場合はコメントを外す
# from google.colab import files; files.download(out_path)

## 6. 結果の可視化

**このランタイムで実際に測定できたデバイス・フレームワークだけ**を表示します。

In [ ]:
def series(model, key_filter, xkey):
    """(device, framework) ごとに、x の値と warm median の列を作る"""
    out = {}
    for r in results:
        if r["model"] == model and key_filter(r):
            out.setdefault((r["device"], r["framework"]), []).append((r[xkey], r["warm_median_s"]))
    return {k: sorted(v) for k, v in out.items()}

LINESTYLE = {"Python": ":", "NumPy": "-", "PyTorch": "--", "JAX": "-."}
MARKER = {"Python": "x", "NumPy": "o", "PyTorch": "s", "JAX": "^"}


def plot_lines(xkey, key_filter, xlabel, title):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
    for ax, model in zip(axes, ["ANN", "LUT"]):
        for (device, fw), pts in series(model, key_filter, xkey).items():
            xs, ys = zip(*pts)
            ax.loglog(xs, np.array(ys) * 1e3, LINESTYLE[fw], marker=MARKER[fw], ms=6,
                      color=COLORS.get(device, COLORS["neutral"]), label=f"{device} / {fw}")
        ax.set_title(f"{model}: {title}"); ax.set_xlabel(xlabel)
    axes[0].set_ylabel("warm median time [ms]"); axes[1].legend(fontsize=8)
    plt.tight_layout(); plt.show()


# 6-1. バッチサイズと実行時間
plot_lines("batch", lambda r: r["n"] == BASE["n"], "batch size B", "time vs batch (n=m=64, nt=16, nc=6)")
# 6-2. 入力次元と実行時間
plot_lines("n", lambda r: r["batch"] == SWEEP_BATCH, "input dim n (= m)", f"time vs input dim (B={SWEEP_BATCH})")

In [ ]:
# 6-3. デバイス別の実行時間（ANN と LUT、バッチ 256・n = 64）と、6-4. 初回とウォームアップ後の違い
sel = [r for r in results if r["batch"] == 256 and r["n"] == BASE["n"] and r["framework"] != "Python"]
labels = sorted({(r["device"], r["framework"]) for r in sel})
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.8))

width = 0.38
for j, model in enumerate(["ANN", "LUT"]):
    vals = [next((r["warm_median_s"] * 1e3 for r in sel if (r["device"], r["framework"]) == lab and r["model"] == model), np.nan)
            for lab in labels]
    ax1.bar(np.arange(len(labels)) + (j - 0.5) * width, vals, width, color=COLORS[model], label=model)
ax1.set_xticks(range(len(labels))); ax1.set_xticklabels([f"{d}\n{f}" for d, f in labels])
ax1.set_yscale("log"); ax1.set_ylabel("warm median [ms]"); ax1.set_title("ANN vs LUT by device (B=256, n=m=64)")
ymin, ymax = ax1.get_ylim(); ax1.set_ylim(ymin, ymax * 6)   # 凡例を置く場所を空ける（log 軸）
ax1.legend(loc="upper left", ncol=2)

sel_lut = [r for r in sel if r["model"] == "LUT"]
names = [f"{r['device']}\n{r['framework']}" for r in sel_lut]
ax2.bar(np.arange(len(sel_lut)) - 0.2, [r["first_s"] * 1e3 for r in sel_lut], 0.4,
        color=COLORS["neutral"], label="first call")
ax2.bar(np.arange(len(sel_lut)) + 0.2, [r["warm_median_s"] * 1e3 for r in sel_lut], 0.4,
        color=COLORS["LUT"], label="after warm-up (median)")
jx = [i for i, r in enumerate(sel_lut) if r.get("first_with_compile_s") is not None]
if jx:   # JAX は「jit してすぐ呼んだ初回（コンパイル込み）」も表示
    ax2.scatter(np.array(jx) - 0.2, [sel_lut[i]["first_with_compile_s"] * 1e3 for i in jx],
                marker="v", s=60, color="black", zorder=3, label="JAX first call incl. compile")
ax2.set_xticks(range(len(sel_lut))); ax2.set_xticklabels(names)
ax2.set_yscale("log"); ax2.set_ylabel("time [ms]"); ax2.set_title("LUT: first call vs warm (B=256)")
ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()

### 6-5. 複数のランタイムの結果をまとめる（任意）

CPU・GPU・TPU の各ランタイムで保存した CSV を `notebooks/results/` に置いてから実行すると、デバイスをまたいで比べられます。
存在するファイルだけを読みます（**測っていないデバイスの値は作りません**）。

In [ ]:
import glob

merged = []
for path in sorted(glob.glob("notebooks/results/04_benchmark_*.csv")):
    with open(path) as f:
        for r in csv.DictReader(f):
            r["runtime"] = os.path.basename(path)
            merged.append(r)
print("files:", sorted({r["runtime"] for r in merged}))
# TODO：merged を使って、ランタイムごとの ANN・LUT の warm_median_s を比較するグラフを作る

## 7. 結果の解釈（読み方のヒント）

- **小さいバッチ（B = 1）**：GPU・TPU は命令を出すだけで一定の時間（起動のオーバーヘッド）がかかるため、CPU（NumPy）の方が速いことがよくあります。
- **大きいバッチ**：ANN（行列積）では GPU・TPU が有利になりやすい。LUT は比較・gather・和の組合せなので、ANN ほど差が出ないことがあります。どうなったかは**自分の測定結果**で確かめてください。
- **初回とウォームアップ後**：JAX の初回はコンパイルを含むので、計算本体より何桁も遅いことがあります。PyTorch の GPU も初回は CUDA の初期化を含むことがあります。
- **データ転送**：GPU・TPU で計算する前に、データを CPU から送る時間がかかります。計算本体が速くても、毎回転送すると全体は遅くなることがあります。
- **純 Python と NumPy の差**：同じ CPU でも、実装方法（for ループかバッチ演算か）で何桁も変わります。「デバイスの差」と「実装の差」を混同しないこと。

> ### この比較の限界
> - 測ったのは **Colab の共有マシン上の実行時間**です。同じランタイムでも、混雑状況・割り当てられた機種で変わります。
> - GPU・TPU は ANN の行列演算に最適化されたハードウェアです。SNN 専用（ニューロモルフィック）ハードウェアとは設計思想が違います。
> - **消費電力は測っていません。この結果から SNN や LUT の電力効率は結論できません。**

## 8. 学生向け TODO

- [ ] **TODO 1**：CPU・GPU・TPU の 3 つのランタイムで、それぞれこの Notebook を実行し CSV を保存する（TPU は使えない場合があります。使えなかった場合はそう記録する）。
- [ ] **TODO 2**：`BATCH_SIZES` を変えて、GPU が CPU より速くなるバッチサイズの境目を探す。
- [ ] **TODO 3**：`BASE` の `nt`・`nc` を変えて、LUT の時間がどう変わるか調べる（`MAX_BYTES` に注意）。
- [ ] **TODO 4**：入力の転送時間 `input_transfer_s` と計算本体 `warm_median_s` を比べ、「毎回データを送る場合」の合計時間を考える。
- [ ] **TODO 5**：`DTYPE` を `np.float16`（GPU）や bfloat16（TPU、JAX）に変えた場合を調べる（結果の一致の許容誤差も見直す）。
- [ ] **TODO 6**：6-5 で、ランタイムをまたいだ比較グラフを作る。

## 9. 考察課題

1. ANN と LUT で、バッチサイズを大きくしたときの時間の増え方はどう違いましたか。行列積と gather の違いから説明してください。
2. JAX のコンパイル時間と、ウォームアップ後の実行時間は何倍違いましたか。実際の推論サービスでは、この差はどのように扱われると思いますか。
3. 「GPU で速かった」「TPU で速かった」という結果から、SNN 専用ハードウェアの電力効率について**言えること・言えないこと**を分けて書いてください。
4. 論文 Table IV の「保存容量（footprint）は大きいが、読み出し量（bandwidth）は小さい」という見積もりは、この実験で確かめられましたか。確かめるには何を測る必要がありますか。

## 10. 参考文献

- E. Izhikevich, "Spiking Manifesto", arXiv:2512.11843, 2025. <https://arxiv.org/abs/2512.11843>
- 参考 Notebook：`ai-programming-2026-team-project-1-starter/A_cpu_gpu_tpu/A_cpu_gpu_tpu_benchmark/starter.ipynb`（MIT License, Copyright (c) 2026 Daigo Takeuchi）
- PyTorch ドキュメント：CUDA semantics（非同期実行と同期）<https://pytorch.org/docs/stable/notes/cuda.html>
- JAX ドキュメント：Asynchronous dispatch（`block_until_ready`）、Just-in-time compilation <https://docs.jax.dev/>
- E3 リポジトリの `README.md`（CPU・GPU・TPU 比較、論文との対応関係）